In [ ]:
import json
from pathlib import Path

In [ ]:
# 경로 설정
base_dir = "/Users/sepgom/gom_programmer/멋쟁이사자 NLP/실전02/my_project/processed_data"

raw_train = f"{base_dir}/train.json"
raw_valid = f"{base_dir}/val.json"

train_out = "train_con.json"
valid_out = "valid_con.json"

In [ ]:
SYSTEM_PROMPT = "당신은 사용자의 지시를 이해하고 적절한 형식으로 한국어 응답을 생성하는 도우미이다."

In [ ]:
def load_raw_data(path: str):
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f"No files: {path}")

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("json should be a list type")

    return data

In [ ]:
def convert_sample(sample: dict):
    instruction = str(sample.get("instruction", "")).strip()
    input_text = str(sample.get("input", "")).strip()
    output_text = str(sample.get("output", "")).strip()

    if not instruction or not input_text or not output_text:
        return None

    converted = {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"지시: {instruction}\n\n입력:\n{input_text}"
            },
            {
                "role": "assistant",
                "content": output_text
            }
        ]
    }

    return converted

In [ ]:
def save_jsonl(path: str, data: list):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [ ]:
def convert_and_save(input_path: str, output_path: str, label: str):
    raw_data = load_raw_data(input_path)
    converted_data = []
    skipped = 0

    for sample in raw_data:
        converted = convert_sample(sample)
        if converted is None:
            skipped += 1
            continue
        converted_data.append(converted)

    save_jsonl(output_path, converted_data)

    print(f"[{label}] 전체: {len(raw_data)} | 변환: {len(converted_data)} | 스킵: {skipped} -> {output_path}")
    return len(raw_data), len(converted_data), skipped

In [ ]:
def main():
    print("=" * 50)
    convert_and_save(raw_train, train_out, "Train")
    convert_and_save(raw_valid, valid_out, "Valid")
    print("=" * 50)
    print("변환 완료!")

In [ ]:
if __name__ == "__main__":
    main()